# GRPO fine-tune of Phi-4-mini on the pharma cold-chain environment (Kaggle)

**Goal: a first positive signal.** The trained model should read the news better than the untrained one,
measured on seeds it never saw.

**What the model learns.** The `adjust` format: on each day the baseline rule places a default order, and the model answers
only with **extra safety days per product (0 to +10)**. Training uses only **news days** from the **random scenario**
with **level-2 news**.

**Reward** (`training/grpo_reward.py`, spec in `training/REWARD.md`): the answer is applied today, and then the baseline
plays 15 days in each of 4 shared possible futures. The reward is the gain over answering 0, in points
(×100/15). Format penalties are −1 for no JSON and −0.25 for a value out of range. Each prompt gets a group of
8 answers, with Dr. GRPO (`scale_rewards=False`).

**What the notebook does**

| Part | What happens |
|---|---|
| 0–2 | Setup, config, model + LoRA |
| 3 | Training prompts (train seeds 0–8999) and the probe exam (held-out seeds 9000–9099) |
| 4 | **Before training:** test-seed probe and full episodes with the untrained model |
| 5–6 | GRPO training with health metrics, and a probe exam on each checkpoint |
| 7 | Save the LoRA adapter |
| 8 | **After training:** test-seed probe, full episodes, news-0 and fixed-scenario checks, the verdict |
| 9 | Qualitative look, GGUF export for Ollama, and a results zip |

**Kaggle settings.**
- **Accelerator:** **GPU T4 ×2** (recommended). It also runs on one GPU (T4 ×1 or P100), just more slowly.
- **Internet:** on.
- **Run mode:** *Save Version → Save & Run All* (runs in the background, up to 12 h).

**With two GPUs**, training runs on GPU 0 and all evaluation runs on GPU 1 in a background process
(`training/eval_worker.py`), so training never waits for it:

| | GPU 0 (this notebook) | GPU 1 (background worker) |
|---|---|---|
| Part 4 | starts training after a ~6 min check that the worker runs | untrained model: probe exams, 30 + 9 episodes (~1.7 h) |
| Part 6 | GRPO training | probe exam on every checkpoint (every 25 steps) |
| Part 8 | test probe, half the episodes, fixed check | other half of the episodes, news-0 check |

GRPO itself stays on one GPU: Unsloth's free version trains on a single GPU, and splitting a batch of 16 answers
across two GPUs would gain little. Moving the evaluation off the training GPU saves ~4 h of model calls.

**Time:** one answer takes ~2.4 s on a T4. Training stops early by itself if the evaluation after it would not fit in
`SESSION_HOURS`, and it prints its seconds per step every 10 steps.

Everything is written to `/kaggle/working/` (shown under *Output*).

## 0. Setup: install, get the code

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Install the libraries.
#
# Why the Pillow pin: Kaggle has already loaded Pillow (an image library) into this kernel before any cell runs.
# If pip replaced Pillow on disk, the kernel would mix the old version in memory with the new one on disk, and
# `import unsloth` would fail with "The _imaging extension was built for another version of Pillow".
# (Unsloth would pull Pillow 13.x, while gradio, needed by openenv-core, wants below 13.)
# So both installs are told to keep Pillow at exactly the version this kernel already has.
# ---------------------------------------------------------------------------------------------------------------
import importlib.metadata, os, sys, tempfile

# The Pillow version to keep: the one in memory if Kaggle loaded it, otherwise the one installed.
loaded = sys.modules.get("PIL")
PILLOW = getattr(loaded, "__version__", None) or importlib.metadata.version("pillow")

# A pip "constraints" file: "whatever you install, Pillow must stay at this version".
CONSTRAINTS = os.path.join(tempfile.gettempdir(), "pip_constraints.txt")
with open(CONSTRAINTS, "w") as f:
    f.write(f"pillow=={PILLOW}\n")
print(f"keeping Pillow {PILLOW}" + (" (already loaded in this kernel)" if loaded else ""))

# Unsloth: fast 4-bit loading + LoRA training. It also brings matching torch / transformers / TRL / PEFT.
!pip install -q -c {CONSTRAINTS} unsloth
# openenv-core: the environment framework our simulator is built on. plotly: used by the repo's dashboard.
!pip install -q -c {CONSTRAINTS} "openenv-core>=0.2.1" plotly

**Pillow check.** This cell confirms that Pillow on disk works (tested in a separate process, together with
torchvision). It also confirms that Pillow on disk is the same version as the one already loaded in this kernel. If the
copy on disk is broken, the cell reinstalls one clean copy of the loaded version. If the two versions differ, the cell
stops and asks for a kernel restart, which is the only fix for that.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Check Pillow after the installs, and repair it if needed.
#
# The test runs in a *separate* Python process, so this kernel never loads a broken Pillow itself.
# Two checks:
#   1. Pillow on disk imports cleanly (together with torchvision, which Unsloth loads).
#   2. Pillow on disk is the same version as the one already in this kernel's memory.
# ---------------------------------------------------------------------------------------------------------------
import glob, shutil, site, subprocess


def pillow_on_disk():
    """(works, version) for the Pillow a fresh Python process would load."""
    test = ("import PIL, PIL.Image\n"
            "try:\n    import torchvision\nexcept ModuleNotFoundError:\n    pass\n"
            "print(PIL.__version__)")
    r = subprocess.run([sys.executable, "-c", test], capture_output=True, text=True)
    last = (r.stdout + r.stderr).strip().splitlines()
    return r.returncode == 0, last[-1] if last else ""   # the last line: the version, or the error


ok, disk = pillow_on_disk()
if not ok:
    # Broken install (files from two versions mixed): remove every copy, then install one clean copy.
    print(f"Pillow on disk is broken ({disk}); reinstalling one clean copy of {PILLOW}")
    for _ in range(2):                       # twice: there can be more than one installed copy
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "pillow"], capture_output=True)
    for folder in site.getsitepackages() + [site.getusersitepackages()]:      # leftovers pip did not remove
        for path in glob.glob(os.path.join(folder, "PIL")) + glob.glob(os.path.join(folder, "[Pp]illow-*")):
            shutil.rmtree(path, ignore_errors=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", f"pillow=={PILLOW}"], check=True)
    ok, disk = pillow_on_disk()
assert ok, f"Pillow is still broken: {disk}"

# Memory vs disk: a mismatch cannot be fixed without restarting the kernel.
in_memory = getattr(sys.modules.get("PIL"), "__version__", None)
if in_memory and in_memory != disk:
    raise RuntimeError(f"This kernel has Pillow {in_memory} loaded but {disk} is on disk. "
                       "Run -> Restart & clear cell outputs, then run all cells from the top again.")
print(f"Pillow OK: {disk} on disk" + (f", {in_memory} in this kernel" if in_memory else ""))

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Get the project code from GitHub, and decide how to use the GPUs.
# ---------------------------------------------------------------------------------------------------------------
import os, subprocess, sys, time

NOTEBOOK_START = time.time()   # the time budget counts from here (Kaggle stops a session after 12 h)

REPO = "BishnuGanguly/pharma_coldstorage_env"
BRANCH = "claude/awesome-cori-1hwdfm"      # switch to "main" once the training PR is merged
WORK = "/kaggle/working"                   # Kaggle keeps this folder as the notebook's Output
CODE_DIR = "/kaggle/working/pharma_coldstorage_env"

# A private repo needs a GitHub token saved as the Kaggle secret GITHUB_TOKEN (Add-ons -> Secrets).
# A public repo needs nothing: the token stays empty.
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"

# Clone once (a re-run of this cell keeps the existing copy), and show which commit we run.
if not os.path.isdir(CODE_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, CODE_DIR], check=True)
print(subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

# Work from inside the repo, so `import eval`, `import tasks`, `from training import ...` find the project's files.
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)

# GPUs. This kernel (training) always uses GPU 0. With a second GPU, all evaluation runs there in a background
# process (training/eval_worker.py), so training never has to wait for it.
# CUDA_VISIBLE_DEVICES must be set before torch is imported (next cell); afterwards it has no effect.
USE_SECOND_GPU = True          # set to False to run everything on GPU 0
try:
    N_GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
except FileNotFoundError:      # no NVIDIA driver: no GPU at all
    N_GPUS = 0
TWO_GPUS = USE_SECOND_GPU and N_GPUS >= 2
os.environ["CUDA_VISIBLE_DEVICES"] = "0"       # this kernel sees only GPU 0
print(f"{N_GPUS} GPU(s): " + ("training on GPU 0, evaluation on GPU 1" if TWO_GPUS else "everything on one GPU"))

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Imports.
# ---------------------------------------------------------------------------------------------------------------
# Unsloth must be imported before transformers / TRL: it patches them to be faster and use less memory.
import unsloth  # noqa: F401
from unsloth import FastLanguageModel, is_bfloat16_supported

import json, math, random, statistics as st, threading, time
from collections import deque
from dataclasses import fields
from pathlib import Path

import torch
from transformers import TrainerCallback     # hooks that run during training (our monitoring)
from datasets import Dataset                 # the table of training prompts TRL reads
from trl import GRPOConfig, GRPOTrainer      # TRL's GRPO implementation

# The project's own code (all of it runs on CPU, except training/gpu.py):
import eval as E                             # running episodes, statistics, paired comparisons
from tasks import TASK_REGISTRY, TEST_SEEDS  # the 3 tasks; the test seeds (never used for training)
from training import episodes as EP          # many episodes at once with one batched model call per day
from training import eval_worker as W        # the GPU 1 worker (its jobs can also run here)
from training import probe as P              # the probe exam: fixed news days, scored with the reward
from training.gpu import make_generate, prepare_tokenizer   # answering prompts with the model
from training.grpo_reward import GRPOReward, build_state, dataset_rows, news_day_prompts   # the reward
from training.monitor import MonitoredReward # the reward + training-health numbers

# Versions, useful when something breaks.
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
import trl, transformers, peft
print("trl", trl.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)

## 1. Config

Every setting is in one place. The defaults follow what we measured: temperature 1.0 gave enough variety in the
diversity check; groups of 8; 15-day horizon; 4 futures.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Config: every setting in one place. Change these, not the code below.
# ---------------------------------------------------------------------------------------------------------------
OUT = Path(WORK) / "grpo_run"          # every result, log and checkpoint goes here
OUT.mkdir(parents=True, exist_ok=True)

# --- Model ---
MODEL_NAME = "unsloth/Phi-4-mini-instruct"   # 3.8B parameters, loaded in 4-bit
MAX_PROMPT_TOKENS = 3072        # our prompts are ~2,000-2,600 tokens (system prompt + report + 5-day history)
MAX_COMPLETION_TOKENS = 96      # the answer is a JSON object of 5 numbers: ~40-60 tokens
LORA_RANK = 16                  # width of the LoRA "correction" matrices: bigger = more capacity and more memory

# --- Training prompts ---
TASKS = sorted(TASK_REGISTRY)                 # supply_chain_broken, flu_season, epidemic_two_wave
TRAIN_SEED_POOL = range(0, 9000)              # train split, minus the probe seeds 9000-9099
TRAIN_EPISODES_PER_TASK = 120                 # episodes sampled to make the prompt pool (~14 news days each)
MIN_SPREAD = 0.25    # keep news days where the reference answers (0/2/4/7/10) differ by at least this many points

# --- GRPO ---
NUM_GENERATIONS = 8                 # answers per prompt (the group the model is compared within)
PER_DEVICE_BATCH = 8                # answers per forward pass: one group
GRAD_ACCUM = 2                      # -> 2 prompts (16 answers) per optimizer step
TRAIN_STEPS = 300                   # at most 600 prompts / 4,800 scored answers (the time budget may stop earlier)
LEARNING_RATE = 1e-5                # small: RL updates are noisy
BETA_KL = 0.04                      # strength of the pull back towards the original model
TEMPERATURE = 1.0                   # sampling temperature during training (1.0 gave enough variety in the check)
SAVE_EVERY = 25 if TWO_GPUS else 50 # steps between checkpoints; with two GPUs, each one gets a probe exam on GPU 1
SESSION_HOURS = 11.0                # training stops early if needed, so the evaluation after it still fits in 12 h

# --- Monitoring and evaluation ---
PROBE_EVERY = SAVE_EVERY            # steps between probe exams (~4 min each on a T4)
PROBE_COUNT = 100                   # news days in the probe exam
EVAL_SEEDS = 10                     # test seeds per task for full 60-day episodes (3 tasks -> 30 episodes)
CHECK_SEEDS = 3                     # test seeds per task for the news-0 and fixed-scenario checks
GEN_BATCH = 8                       # prompts per generate() call at evaluation (lower it if out of memory)
EXPORT_GGUF = True                  # also export a GGUF file for Ollama at the end (~10-20 min)
SECONDS_PER_ANSWER = 2.5            # measured on a T4 (2.42 s); used to keep enough time for the evaluation

# Fixed random seeds: the same config gives the same prompts and the same order.
random.seed(0)
torch.manual_seed(0)

## 2. Model: Phi-4-mini in 4-bit, with LoRA

`generate()` is used by the probe exam and the full episodes. It answers greedily, so the base and trained models get
compared on exactly the same prompts with no sampling noise. GRPO samples on its own at `TEMPERATURE`.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Load Phi-4-mini in 4-bit and attach the LoRA adapter.
# ---------------------------------------------------------------------------------------------------------------
# 4-bit: the 3.8B weights take ~2.5 GB instead of ~7.6 GB, leaving GPU memory for GRPO's 8 answers per prompt.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_PROMPT_TOKENS + MAX_COMPLETION_TOKENS,   # the longest text (prompt + answer) it must handle
    load_in_4bit=True,
    dtype=None,                     # Unsloth picks float16 / bfloat16 for the GPU
)

# Which weight matrices get a LoRA "correction". Phi-4-mini fuses q/k/v into qkv_proj and gate/up into gate_up_proj;
# other builds of the model split them. So we take whichever names exist in the model.
module_names = {name.split(".")[-1] for name, _ in model.named_modules()}
candidates = ["qkv_proj", "q_proj", "k_proj", "v_proj", "o_proj", "gate_up_proj", "gate_proj", "up_proj", "down_proj"]
targets = [m for m in candidates if m in module_names]
print("LoRA targets:", targets)

# Freeze the 3.8B original weights and add small trainable matrices (~23M numbers, ~0.6% of the model).
# The LoRA starts at zero, so right now the model is still exactly the original Phi-4-mini.
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,                    # rank: width of the narrow middle of each correction
    lora_alpha=LORA_RANK,           # strength: the correction is scaled by alpha / r = 1
    lora_dropout=0,                 # no random dropout (RL is noisy enough already)
    target_modules=targets,
    use_gradient_checkpointing="unsloth",   # recompute some values instead of storing them: less memory
    random_state=0,
)
model.print_trainable_parameters()  # should show ~23M trainable out of ~3.8B
prepare_tokenizer(tokenizer)        # pad on the left, as batched generation needs

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Helpers: answering prompts with the model, and running the GPU 1 worker.
# ---------------------------------------------------------------------------------------------------------------
# generate(list of chats) -> one answer per chat. Greedy (no randomness), so the same model always gives the same
# answer, and base vs trained comparisons are not blurred by sampling luck. (GRPO samples on its own, at TEMPERATURE.)
# It lives in training/gpu.py so the GPU 1 worker uses exactly the same function.
generate = make_generate(model, tokenizer, GEN_BATCH, MAX_COMPLETION_TOKENS)


def back_to_training():
    """generate() switches the model to its fast inference mode; this switches it back before training continues."""
    FastLanguageModel.for_training(model)


def launch_worker(tag, jobs):
    """
    Start training/eval_worker.py on GPU 1 in the background and return its process handle.
    `jobs` is a list of dicts (probe exams, episodes, the checkpoint watch); see that file's docstring.
    The worker writes its results to OUT, its progress to OUT/worker_<tag>.log, and the names of the jobs it has
    finished to OUT/worker_<tag>_status.json.
    """
    spec = {"model": MODEL_NAME, "out": str(OUT), "max_seq_length": MAX_PROMPT_TOKENS + MAX_COMPLETION_TOKENS,
            "gen_batch": GEN_BATCH, "max_new_tokens": MAX_COMPLETION_TOKENS, "tag": tag, "jobs": jobs}
    (OUT / f"worker_{tag}.json").write_text(json.dumps(spec, indent=1))
    log = open(OUT / f"worker_{tag}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": "1"}       # the worker sees only GPU 1
    return subprocess.Popen([sys.executable, "training/eval_worker.py", str(OUT / f"worker_{tag}.json")],
                            stdout=log, stderr=subprocess.STDOUT, env=env, cwd=CODE_DIR)


def worker_log(tag, lines=15):
    """The last lines of a worker's log (shown when something goes wrong)."""
    path = OUT / f"worker_{tag}.log"
    return "\n".join(path.read_text().splitlines()[-lines:]) if path.exists() else "(no log)"


def worker_done(tag):
    """Names of the jobs a worker has finished so far."""
    path = OUT / f"worker_{tag}_status.json"
    return json.loads(path.read_text())["done"] if path.exists() else []


def run_here(jobs):
    """
    Run worker jobs in this kernel on GPU 0 instead: in one-GPU mode, or as a fallback when the worker fails.
    It writes the same output files as the worker, so the rest of the notebook does not care where a job ran.
    """
    for job in jobs:
        t = time.time()
        if job["kind"] == "probe":
            W.probe_job(job, generate, OUT)
        elif job["kind"] == "episodes":
            W.episodes_job(job, generate, OUT)
        print(f"  {job['name']} done in {(time.time() - t) / 60:.0f} min", flush=True)

## 3. Prompts: training pool and probe exam

- **Training prompts:** news days from train seeds 0–8999, random scenario, news level 2. Days where no answer moves the
  reward (reference spread below `MIN_SPREAD`) are dropped, because every answer would score the same. Days where
  *holding back* is right are kept.
- **Probe exam:** 100 news days from seeds 9000–9099, which training never sees. Before training, the probe exam is
  taken again on **test seeds**, so there is a clean before/after on seeds nobody tuned on.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Training prompts: one prompt = one news day of one episode (random scenario, level-2 news, train seeds).
# ---------------------------------------------------------------------------------------------------------------
t0 = time.time()
reward_ref = GRPOReward()

# Sample some train episodes and list every day that has visible news (quiet days teach nothing).
train_seeds = sorted(random.Random("train-seeds").sample(list(TRAIN_SEED_POOL), TRAIN_EPISODES_PER_TASK))
pool = news_day_prompts(TASKS, train_seeds)
random.Random("train-order").shuffle(pool)          # mix tasks and days

# How many prompts training uses: each step takes PER_DEVICE_BATCH x GRAD_ACCUM answers, 8 per prompt.
needed = TRAIN_STEPS * GRAD_ACCUM * PER_DEVICE_BATCH // NUM_GENERATIONS

# Keep days where the answer matters. If +0, +2, +4, +7 and +10 days on the news products all score about the
# same, all 8 answers would get the same reward and GRPO would learn nothing from that prompt. Days where holding
# back (+0) is clearly best are kept: they teach "don't overreact".
kept, dropped = [], 0
for p in pool:
    refs = P.reference_rewards(reward_ref, p)       # {0: 0.0, 2: ..., 4: ..., 7: ..., 10: ...} in points
    if max(refs.values()) - min(refs.values()) >= MIN_SPREAD:
        kept.append(p)
    else:
        dropped += 1
    if len(kept) >= int(needed * 1.2) + 50:          # a few more than needed, then stop (scoring takes time)
        break
print(f"{len(pool)} news days in the pool; kept {len(kept)} (dropped {dropped} where the answer hardly matters); "
      f"training needs {needed}. ({time.time() - t0:.0f}s)")
print("by task:", {t: sum(p.task == t for p in kept) for t in TASKS})

# The table TRL reads: a "prompt" column (the chat messages the model sees) plus task / seed / day / scenario /
# news, which TRL passes on to the reward function so it can rebuild that day in the simulator.
train_dataset = Dataset.from_list(dataset_rows(kept))
print(train_dataset)

# Prompt length check: GRPO cuts prompts longer than MAX_PROMPT_TOKENS, which could cut off the news.
lengths = [len(tokenizer.apply_chat_template(r["prompt"], tokenize=True, add_generation_prompt=True))
           for r in train_dataset.select(range(min(200, len(train_dataset))))]
print(f"prompt tokens: min {min(lengths)}, mean {st.mean(lengths):.0f}, max {max(lengths)} (limit {MAX_PROMPT_TOKENS})")
assert max(lengths) <= MAX_PROMPT_TOKENS, "raise MAX_PROMPT_TOKENS"

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# The probe exams: fixed sets of news days, answered greedily and scored with the GRPO reward.
# ---------------------------------------------------------------------------------------------------------------
t0 = time.time()
# During training: seeds 9000-9099. They belong to the train split but training never uses them (it samples 0-8999).
probe_items = P.build_probe(seeds=P.PROBE_SEEDS, count=PROBE_COUNT)
# Before vs after training: test seeds, which nothing was ever tuned on. salt="test-probe" picks a different set of days.
test_probe_items = P.build_probe(seeds=TEST_SEEDS[:40], count=PROBE_COUNT, salt="test-probe")
print(f"probe exams built in {time.time() - t0:.0f}s")

# For each exam: how much there is to gain (best reference answer per day), and on how many days 0 is the right answer.
for name, items in (("probe (9000-9099)", probe_items), ("test probe", test_probe_items)):
    zero_best = sum(i.zero_is_best for i in items)
    print(f"{name}: {len(items)} days, mean best achievable {st.mean(i.best for i in items):.2f} points, "
          f"holding back is best on {zero_best}")


def run_probe(items, label):
    """
    Take a probe exam with the model in this kernel and print one line:
      reward         mean points per answer (answering 0 everywhere scores exactly 0)
      % of best      total reward / total of the best reference answer per day
      no JSON        answers the reward could not read
      all-zero       answers that added no extra days
      right product  answers adding days whose largest extra is on a product in the news
      insulin habit  days without insulin news where insulin still got extra days
      held back      days where every +N loses, answered with 0
    """
    t = time.time()
    texts = generate([i.messages for i in items])
    results = P.score_probe(items, texts)            # one dict per answer: reward, text, behaviour checks
    summary = P.summarize_probe(items, results)
    print(f"[{label}] reward {summary['reward']:+.3f} | {summary['pct_of_best']:5.1f}% of best | "
          f"no JSON {summary['no_json']:.0%} | all-zero {summary['all_zero']:.0%} | right product "
          f"{summary['right_product']:.0%} | insulin habit {summary['insulin_habit']:.0%} | held back "
          f"{summary['held_back']:.0%}  ({time.time() - t:.0f}s)")
    return results, summary

## 4. Before training: the untrained model

These runs give the "before" numbers: probe exams on the probe and test seeds, 30 full episodes (random scenario), and
9 fixed-scenario episodes. They use the **untrained** model. On GPU 0, the LoRA is still at zero, so the model is
exactly the base model. On GPU 1, the worker loads the base model itself.

- **Two GPUs:** the worker does all of this on GPU 1 and then starts its probe-exam watch. This cell waits only for the
  worker's first job (~6 min) to confirm it runs, then moves on, so training starts on GPU 0 right away. If the
  worker fails, the notebook switches to one-GPU mode.
- **One GPU:** this runs here (~2 h).

Results go to files (`base_*.json`, `base_*.jsonl` in `OUT`), and part 8 reads them from there.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Part 4: the untrained ("base") model, measured before any training. These are the "before" numbers.
# ---------------------------------------------------------------------------------------------------------------
eval_seeds = list(TEST_SEEDS[:EVAL_SEEDS])     # test seeds for the full episodes (the same ones after training)
check_seeds = list(TEST_SEEDS[:CHECK_SEEDS])   # test seeds for the fixed-scenario and news-0 checks

# The work, written as "jobs" so it can run either on GPU 1 (the worker) or here (run_here) with the same output files.
PROBE_JOB = {"kind": "probe", "seeds": list(P.PROBE_SEEDS), "count": PROBE_COUNT, "salt": "probe"}
TEST_PROBE_JOB = {"kind": "probe", "seeds": list(TEST_SEEDS[:40]), "count": PROBE_COUNT, "salt": "test-probe"}
BASE_JOBS = [
    {**PROBE_JOB, "name": "base_probe"},              # step 0 of the probe curve        -> base_probe.json
    {**TEST_PROBE_JOB, "name": "base_test_probe"},    # "before" of the test probe       -> base_test_probe.json
    {"kind": "episodes", "name": "base_random_news2", "tasks": TASKS, "seeds": eval_seeds, "news": 2,
     "scenario": "random", "label": "phi4-mini-base"},   # the main comparison           -> base_random_news2.jsonl
    {"kind": "episodes", "name": "base_fixed_news2", "tasks": TASKS, "seeds": check_seeds, "news": 2,
     "scenario": "fixed", "label": "phi4-mini-base"},    # the fixed-scenario check      -> base_fixed_news2.jsonl
]
# The worker's last job during training: probe every checkpoint the trainer saves, until training_done appears.
WATCH_JOB = {**PROBE_JOB, "kind": "watch", "name": "probe_watch", "checkpoints": str(OUT / "checkpoints"),
             "done_flag": str(OUT / "training_done"), "history": "probe_history.jsonl"}
# Start clean if this cell is re-run.
(OUT / "training_done").unlink(missing_ok=True)
(OUT / "probe_history.jsonl").unlink(missing_ok=True)

worker1 = None
if TWO_GPUS:
    # Start the worker on GPU 1, then wait only for its first job (~6 min with the model load) to be sure it works.
    # Everything after that runs in the background while training starts on GPU 0.
    worker1 = launch_worker("gpu1_before", BASE_JOBS + [WATCH_JOB])
    t0 = time.time()
    while "base_probe" not in worker_done("gpu1_before") and worker1.poll() is None and time.time() - t0 < 1800:
        time.sleep(15)
    if "base_probe" in worker_done("gpu1_before"):
        print(f"GPU 1 worker is running: first probe exam done in {(time.time() - t0) / 60:.0f} min. "
              "It keeps evaluating the base model in the background while training starts.")
        print(worker_log("gpu1_before", 3))
    else:
        # The worker crashed (e.g. it could not load the model) or took over 30 min: do everything on GPU 0 instead.
        print("The GPU 1 worker failed or is too slow; switching to one-GPU mode. Its log:\n" + worker_log("gpu1_before"))
        if worker1.poll() is None:
            worker1.kill()
        worker1, TWO_GPUS = None, False
        PROBE_EVERY = 50

if not TWO_GPUS:
    # One GPU: run the base jobs here (~2 h), and measure the real speed for the time budget.
    t0 = time.time()
    run_here(BASE_JOBS)
    SECONDS_PER_ANSWER = (time.time() - t0) / (2 * PROBE_COUNT + 60 * len(TASKS) * (len(eval_seeds) + len(check_seeds)))
    print(f"{SECONDS_PER_ANSWER:.2f}s per answer")


def after_training_minutes():
    """
    Minutes to keep free after training for parts 7-9, so the time budget can stop training in time.
    Two GPUs share the evaluation, so the slower of the two shares counts; plus saving and the GGUF export.
    """
    half = math.ceil(len(eval_seeds) / 2)
    episodes = 60 * len(TASKS)                 # answers per seed: 60 days x 3 tasks
    if TWO_GPUS:
        here = PROBE_COUNT + episodes * (half + len(check_seeds))                         # test probe, half, fixed
        there = episodes * (len(eval_seeds) - half + len(check_seeds)) + 2 * PROBE_COUNT  # other half, news 0, last probe
        answers = max(here, there)
    else:
        answers = PROBE_COUNT + episodes * (len(eval_seeds) + 2 * len(check_seeds))
    return answers * SECONDS_PER_ANSWER / 60 + (25 if EXPORT_GGUF else 5) + 5


AFTER_TRAINING_MINUTES = after_training_minutes()
print(f"time kept for after training: ~{AFTER_TRAINING_MINUTES:.0f} min")

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# The reference agents: rule-based, no model, a few seconds on CPU. Played on the same 30 test episodes as the
# model, so every comparison in part 8 is like for like.
# ---------------------------------------------------------------------------------------------------------------
refs = {
    # The order rule alone, ignoring the news. The model answering 0 every day plays exactly like this.
    "baseline": [E.run_episode("baseline", t, s, news=2, scenario="random") for t in TASKS for s in eval_seeds],
    # +3 extra days on every product, every day: beating it shows the model reads the news, not just stocks more.
    "plus3_everywhere": EP.run_rule_episodes(EP.FlatExtraAgent(3), "plus3_everywhere", TASKS, eval_seeds),
    # Extra days from a formula using the exact hidden events: what perfect news reading is worth.
    "perfect_news": [E.run_episode("baseline_news", t, s, news=2, scenario="random") for t in TASKS for s in eval_seeds],
    # Knows the future: the ceiling, used to report the share of the gap the model closes.
    "oracle": [E.run_episode("oracle", t, s, news=2, scenario="random") for t in TASKS for s in eval_seeds],
}
for name, recs in refs.items():
    print(f"{name:18s} {E.summarize(recs)['ALL']['mean']:.4f}")
# With two GPUs, the base episodes are still running on GPU 1 at this point; part 8 reads them later.
if (OUT / "base_random_news2.jsonl").exists():
    print(f"{'base phi4-mini':18s} {E.summarize(E.read_jsonl(OUT / 'base_random_news2.jsonl'))['ALL']['mean']:.4f}")

## 5. Reward and training-health monitoring

`MonitoredReward` gives exactly the GRPO reward and also records health numbers. These are logged every step
next to TRL's own `reward`, `reward_std`, `kl`, `completion_length` and `frac_reward_zero_std`.

| Metric | Healthy | Worrying |
|---|---|---|
| `reward_ma20` (moving average) | rising slowly | flat for 100+ steps |
| `health/zero_std_groups` | below ~0.5 | near 1: groups teach nothing (raise the temperature) |
| `health/all_zero` | drifting to ~0.3–0.5 | 1.0: collapsed to "always 0" |
| `health/no_json` | goes to ~0 | rising |
| `kl` | small, slowly growing | jumps: lower `LEARNING_RATE` |
| `completion_length` | short, stable | growing: rambling |
| `health/right_product` | rising | falling |

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# The reward function, and three "callbacks": small hooks the trainer calls during training.
#   HealthCallback      logs the health numbers every step and prints a summary every 5 steps
#   ProbeCallback       probe exam every PROBE_EVERY steps (two GPUs: prints the GPU 1 worker's results)
#   TimeBudgetCallback  stops training early so the evaluation after it still fits in the session
# ---------------------------------------------------------------------------------------------------------------

# The GRPO reward (training/grpo_reward.py): for each answer, play the next 15 days in 4 possible futures with the
# simulator and score the gain over answering 0, in points. MonitoredReward returns exactly the same rewards and
# also records how the answers look (no JSON, all zero, right product, groups with no spread).
reward_fn = MonitoredReward()


class HealthCallback(TrainerCallback):
    """Adds the health numbers and a 20-step moving-average reward to every log, and keeps them all."""

    def __init__(self):
        self.recent = deque(maxlen=20)    # the last 20 rewards, for the moving average
        self.history = []                 # every logged row, for the plots

    def on_log(self, args, state, control, logs=None, **kwargs):
        # The trainer calls this whenever it logs (every step, logging_steps=1). Skip logs that are not training steps.
        if logs is None or "loss" not in logs and "reward" not in logs:
            return
        logs.update(reward_fn.pop_stats())          # health numbers of the answers scored since the last log
        if "reward" in logs:
            # One step's reward is noisy (only 2 prompts); the 20-step average shows the trend.
            self.recent.append(logs["reward"])
            logs["reward_ma20"] = sum(self.recent) / len(self.recent)
        row = {"step": state.global_step, **{k: v for k, v in logs.items() if isinstance(v, (int, float))}}
        self.history.append(row)
        if state.global_step % 5 == 0:
            # TRL's own numbers (reward, kl, completion_length) next to ours (health/...).
            keys = ["reward", "reward_ma20", "kl", "completion_length", "health/zero_std_groups",
                    "health/all_zero", "health/no_json", "health/right_product"]
            print(f"step {state.global_step:4d} | " + " | ".join(
                f"{k.split('/')[-1]} {row[k]:.3f}" for k in keys if k in row and row[k] == row[k]))  # x == x: not NaN
        with open(OUT / "train_log.jsonl", "a") as f:           # kept on disk too, in case the session ends
            f.write(json.dumps(row) + "\n")


def read_probe_history():
    """Every probe exam so far: one row per checkpoint/step, written by the worker (or by ProbeCallback)."""
    path = OUT / "probe_history.jsonl"
    return [json.loads(line) for line in path.read_text().splitlines()] if path.exists() else []


def probe_line(row):
    return (f"[probe, step {row['step']}] reward {row['reward']:+.3f} | {row['pct_of_best']:5.1f}% of best | "
            f"right product {row['right_product']:.0%} | insulin habit {row['insulin_habit']:.0%} | "
            f"held back {row['held_back']:.0%} | all-zero {row['all_zero']:.0%}")


class ProbeCallback(TrainerCallback):
    """
    The probe exam (held-out seeds 9000-9099, greedy) every PROBE_EVERY steps: the clearest sign of whether
    training is working, because the same 100 days are answered without sampling noise every time.
    Two GPUs: the worker on GPU 1 runs it on each checkpoint; this only prints its results as they arrive.
    One GPU: it runs here, which pauses training for ~4 minutes each time.
    """

    def __init__(self):
        self.shown, self.warned = 0, False

    def on_step_end(self, args, state, control, **kwargs):
        if TWO_GPUS:
            rows = read_probe_history()
            for row in rows[self.shown:]:                # print only the new results
                print(probe_line(row))
            self.shown = len(rows)
            # A crashed worker must not stop training, but it should not go unnoticed either.
            if worker1 is not None and worker1.poll() not in (None, 0) and not self.warned:
                self.warned = True
                print("WARNING: the GPU 1 worker stopped; no more probe exams. Log:\n" + worker_log("gpu1_before"))
            return
        if state.global_step % PROBE_EVERY:
            return
        _, summary = run_probe(probe_items, f"step {state.global_step}")
        with open(OUT / "probe_history.jsonl", "a") as f:
            f.write(json.dumps({"step": state.global_step, **summary}, default=float) + "\n")
        back_to_training()                               # generate() switched the model to inference mode


class TimeBudgetCallback(TrainerCallback):
    """
    Stops training early when the next steps would not leave AFTER_TRAINING_MINUTES before SESSION_HOURS, so the
    evaluation always runs. The LoRA trained so far is kept; the stop is printed.
    """

    def __init__(self):
        self.last, self.intervals, self.stopped_at = None, deque(maxlen=PROBE_EVERY), None

    def on_step_end(self, args, state, control, **kwargs):
        # Measure how long steps take (the recent average, which includes any probe exams run here).
        now = time.time()
        if self.last is not None:
            self.intervals.append(now - self.last)
        self.last = now
        if not self.intervals:
            return
        per_step = sum(self.intervals) / len(self.intervals)
        # Time left for training = session length - time used so far - time the evaluation needs.
        left = SESSION_HOURS * 3600 - (now - NOTEBOOK_START) - AFTER_TRAINING_MINUTES * 60
        if state.global_step % 10 == 0:
            print(f"   time: {per_step:.0f}s per step | {left / 3600:.1f} h left for training "
                  f"(~{max(left, 0) / per_step:.0f} more steps)")
        if left < 2 * max(self.intervals):            # not enough time for two more (slow) steps: stop now
            self.stopped_at = state.global_step
            print(f"Time budget reached: stopping training at step {state.global_step} so the evaluation fits.")
            control.should_training_stop = True       # the trainer finishes this step and returns normally


health_cb, probe_cb, time_cb = HealthCallback(), ProbeCallback(), TimeBudgetCallback()

## 6. GRPO training

The settings are set by name, and any your TRL version does not know is reported and skipped. With
`scale_rewards=False` (Dr. GRPO) the advantages stay in reward points, so a group where every answer is within
0.01 points gives almost no push.

If you run out of memory, set `PER_DEVICE_BATCH = 4` and `GRAD_ACCUM = 4`.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# GRPO settings and the trainer.
#
# What one training step does (TRL's GRPOTrainer):
#   1. take 2 prompts (news days) and sample 8 answers to each, at TEMPERATURE
#   2. score all 16 answers with reward_fn (simulator, CPU)
#   3. within each group of 8: advantage = reward - the group's mean reward
#      (above-average answers are made more likely, below-average ones less likely)
#   4. update only the LoRA weights, with a KL penalty (BETA_KL) that keeps the model close to the original
# ---------------------------------------------------------------------------------------------------------------
known = {f.name for f in fields(GRPOConfig)}      # the setting names this TRL version accepts
settings = dict(
    output_dir=str(OUT / "checkpoints"),
    # Learning rate: constant after a 10-step warm-up (a decaying schedule would make steps late in training count less).
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=10,
    weight_decay=0.0,
    max_grad_norm=0.2,                 # clip large updates: RL gradients can spike
    optim="adamw_8bit",                # the optimizer's memory in 8-bit: saves GPU memory
    # Batch: PER_DEVICE_BATCH answers per forward pass, GRAD_ACCUM passes per update -> 16 answers = 2 groups of 8.
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_TOKENS,
    max_completion_length=MAX_COMPLETION_TOKENS,
    temperature=TEMPERATURE,
    beta=BETA_KL,
    # Dr. GRPO: don't divide the advantages by the group's spread. Rewards stay in points, so a group whose answers
    # are all within 0.01 points gives almost no push, instead of having tiny differences blown up.
    scale_rewards=False,
    loss_type="dr_grpo",
    max_steps=TRAIN_STEPS,
    logging_steps=1,                   # log every step (the health callback reads these logs)
    save_steps=SAVE_EVERY,             # checkpoint = the LoRA at that step (+ optimizer state), ~100 MB
    save_total_limit=None if TWO_GPUS else 2,   # keep them all with two GPUs: the GPU 1 worker probes each one
    bf16=is_bfloat16_supported(),      # T4s have no bfloat16: they use float16
    fp16=not is_bfloat16_supported(),
    report_to="none",                  # no Weights & Biases etc.
    seed=0,
    log_completions=False,
)
# TRL renames settings between versions; drop the ones this version does not know instead of crashing.
skipped = sorted(set(settings) - known)
if skipped:
    print("not in this TRL version, skipped:", skipped)
config = GRPOConfig(**{k: v for k, v in settings.items() if k in known})

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],          # TRL calls reward_fn(completions, task=[...], seed=[...], day=[...], ...)
    args=config,
    train_dataset=train_dataset,
    callbacks=[health_cb, probe_cb, time_cb],
)

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Train. This is the long cell: watch the "step ..." lines, the "[probe, step ...]" lines and the "time:" lines.
# ---------------------------------------------------------------------------------------------------------------
t0 = time.time()
back_to_training()
try:
    trainer.train()
finally:
    # Runs even if training stops with an error, so the worker on GPU 1 is never left waiting:
    # 1. save the last weights as a checkpoint too, so the worker also probes the final model;
    #    trainer_state.json is written last: the worker only reads checkpoints that have it (= finished saving)
    final = OUT / "checkpoints" / f"checkpoint-{trainer.state.global_step}"
    if trainer.state.global_step and not (final / "trainer_state.json").exists():
        model.save_pretrained(str(final))
        tokenizer.save_pretrained(str(final))
        (final / "trainer_state.json").write_text(json.dumps({"global_step": trainer.state.global_step}))
    # 2. tell the worker that training is over: it probes the remaining checkpoints, then exits
    (OUT / "training_done").write_text("done")
print(f"training took {(time.time() - t0) / 3600:.1f} h"
      + (f"; stopped early at step {time_cb.stopped_at} by the time budget" if time_cb.stopped_at else ""))

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Training curves (also saved as PNG files in OUT).
# ---------------------------------------------------------------------------------------------------------------
import matplotlib.pyplot as plt

# Top: training health, logged every step (see the table in part 5 for what healthy looks like).
h = health_cb.history
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
panels = [("reward_ma20", "reward (20-step avg)"), ("kl", "KL to the base model"), ("completion_length", "completion length"),
          ("health/zero_std_groups", "groups with no spread"), ("health/all_zero", "answers all 0"),
          ("health/right_product", "largest extra on a news SKU")]
for ax, (key, title) in zip(axes.flat, panels):
    pts = [(r["step"], r[key]) for r in h if key in r and r[key] == r[key]]     # skip missing and NaN values
    if pts:
        ax.plot(*zip(*pts))
    ax.set_title(title); ax.set_xlabel("step")
plt.tight_layout(); plt.savefig(OUT / "training_curves.png", dpi=120); plt.show()

# Bottom: the probe exam over training. Step 0 is the base model (part 4); the others are the checkpoints.
# This is the curve that should rise if training works.
base_probe_file = OUT / "base_probe.json"
ph = ([{"step": 0, **json.loads(base_probe_file.read_text())["summary"]}] if base_probe_file.exists() else []) \
     + read_probe_history()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r["step"] for r in ph], [r["pct_of_best"] for r in ph], marker="o"); ax[0].set_title("probe: % of best achievable")
for key in ("right_product", "insulin_habit", "held_back", "all_zero"):
    ax[1].plot([r["step"] for r in ph], [r[key] for r in ph], marker="o", label=key)
ax[1].legend(); ax[1].set_title("probe: behaviour")
plt.tight_layout(); plt.savefig(OUT / "probe_curves.png", dpi=120); plt.show()

## 7. Save the LoRA adapter (before the long evaluation, in case the session ends)

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Save the trained LoRA adapter (~45 MB): only the small correction matrices, not the 3.8B base weights.
# Loading it later = base model + this folder. The GPU 1 worker loads it from here for its share of part 8.
# ---------------------------------------------------------------------------------------------------------------
ADAPTER_DIR = OUT / "phi4mini_pharma_lora"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("saved", ADAPTER_DIR)

## 8. After training: did it improve?

1. **Test-seed probe exam:** the same 100 test news days as before training, base vs trained, paired.
2. **Full 60-day episodes:** test seeds, random scenario, news 2. Compared with the base model and four references:
   - the baseline;
   - "+3 everywhere";
   - perfect news reading (`baseline_news`, the formula from the exact events);
   - the oracle.

   Paired by seed, with 95% intervals.
3. **Checks:**
   - **news 0:** with no news, the model should play like the baseline;
   - **fixed scenario:** it should do no worse than the base model.

**Success** = trained beats base with the interval above 0, and trained beats "+3 everywhere".

With two GPUs, GPU 1 first finishes its last probe exam. It then plays half of the episodes and the news-0 check with
the saved LoRA, while GPU 0 plays the other half, the fixed-scenario check and the test probe.

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Part 8a: the trained model plays the same test episodes as the base model did in part 4.
#
# The work is split in two equal shares (two GPUs at once, or one after the other on one GPU):
#   GPU 0 (here):  test probe, the first half of the 30 episodes, the fixed-scenario check
#   GPU 1:         the second half of the episodes, the news-0 check
# ---------------------------------------------------------------------------------------------------------------
half = math.ceil(len(eval_seeds) / 2)
TRAINED = dict(adapter=str(ADAPTER_DIR), tasks=TASKS, label="phi4-mini-grpo", kind="episodes")
jobs_here = [{**TRAINED, "name": "trained_random_news2_a", "seeds": eval_seeds[:half], "news": 2,
              "scenario": "random", "save_replies": True},              # save_replies: the model's answers, day by day
             {**TRAINED, "name": "trained_fixed_news2", "seeds": check_seeds, "news": 2, "scenario": "fixed"}]
jobs_gpu1 = [{**TRAINED, "name": "trained_random_news2_b", "seeds": eval_seeds[half:], "news": 2,
              "scenario": "random", "save_replies": True},
             {**TRAINED, "name": "trained_random_news0", "seeds": check_seeds, "news": 0, "scenario": "random"}]
second = {"proc": None}        # the second worker's process, set by the thread below

if TWO_GPUS:
    # GPU 1 may still be probing the last checkpoints. A background thread waits for that worker to finish and then
    # starts the second one, while this kernel already gets on with GPU 0's share below.
    def start_gpu1_when_free():
        if worker1 is not None:
            worker1.wait()          # the first worker exits after probing the last checkpoint
        second["proc"] = launch_worker("gpu1_after", jobs_gpu1)
    gpu1_thread = threading.Thread(target=start_gpu1_when_free, daemon=True)
    gpu1_thread.start()
else:
    jobs_here += jobs_gpu1          # one GPU: do both shares here

t0 = time.time()
# The trained model is still in memory here, so the test probe runs in this kernel.
trained_test_probe, trained_test_probe_summary = run_probe(test_probe_items, "trained, test seeds")
run_here(jobs_here)
print(f"GPU 0 share done in {(time.time() - t0) / 60:.0f} min")

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Part 8b: wait for GPU 1, then load every result (before and after training) from the files.
# ---------------------------------------------------------------------------------------------------------------
if TWO_GPUS:
    gpu1_thread.join()
    exit_code = second["proc"].wait()
    # If the second worker failed, run whatever it did not finish here, so the results are always complete.
    if exit_code != 0 or not all(j["name"] in worker_done("gpu1_after") for j in jobs_gpu1):
        print("The GPU 1 worker failed; running its share here. Log:\n" + worker_log("gpu1_after"))
        run_here([j for j in jobs_gpu1 if j["name"] not in worker_done("gpu1_after")])

# Every job wrote its results to OUT, wherever it ran.
read = lambda name: E.read_jsonl(OUT / f"{name}.jsonl")
trained_eps = read("trained_random_news2_a") + read("trained_random_news2_b")   # the two halves together
trained_fixed, trained_news0 = read("trained_fixed_news2"), read("trained_random_news0")
base_eps, base_fixed = read("base_random_news2"), read("base_fixed_news2")      # from part 4
E.write_jsonl(OUT / "trained_random_news2.jsonl", trained_eps)
base_test_probe = json.loads((OUT / "base_test_probe.json").read_text())["results"]
base_test_probe_summary = json.loads((OUT / "base_test_probe.json").read_text())["summary"]

# Test probe, paired: the same 100 test days, base answer vs trained answer, day by day.
probe_diff = P.compare_probes(trained_test_probe, base_test_probe)
print(f"test probe, trained - base: {probe_diff['mean']:+.3f} points ± {probe_diff['ci95']:.3f} "
      f"(better on {probe_diff['better']}, worse on {probe_diff['worse']} of {probe_diff['n']})")
print(f"episodes: {len(trained_eps)} trained, {len(base_eps)} base (random scenario)")

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Part 8c: episode scores. "Paired" = each trained episode is compared with the other agent on the same task, seed
# and future, so most of the luck cancels out. "± x" is the 95% confidence interval of the mean difference:
# a difference counts as real when mean - x is still above 0.
# ---------------------------------------------------------------------------------------------------------------
def line(name, comp):
    a = comp["ALL"]
    # gap_closed: the share of the reference-to-oracle gap the candidate closes (0 = no better, 1 = perfect play).
    gap = f" | closes {a['gap_closed']:.0%} of the gap to the oracle" if "gap_closed" in a else ""
    return f"{name:34s} {a['mean']:+.4f} ± {a['ci95']:.4f}{gap}"


scores = {"trained phi4-mini": trained_eps, "base phi4-mini": base_eps, **refs}
print("Mean score, random scenario, news 2, test seeds:")
for name, recs in scores.items():
    print(f"  {name:20s} {E.summarize(recs)['ALL']['mean']:.4f}")

print("\nPaired differences (per seed):")
vs_base = E.paired_comparison(trained_eps, base_eps, refs["oracle"])        # the main question: did training help?
vs_plus3 = E.paired_comparison(trained_eps, refs["plus3_everywhere"])      # better than just stocking more?
print(line("trained - base", vs_base))
print(line("trained - +3 everywhere", vs_plus3))
print(line("trained - baseline", E.paired_comparison(trained_eps, refs["baseline"], refs["oracle"])))
print(line("trained - perfect news reading", E.paired_comparison(trained_eps, refs["perfect_news"])))
print(line("base - baseline", E.paired_comparison(base_eps, refs["baseline"], refs["oracle"])))
print("\nper task, trained - base:")
for task, s in vs_base.items():
    if task != "ALL":
        print(f"  {task:22s} {s['mean']:+.4f} ± {s['ci95']:.4f}")

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Part 8d: the safety checks, the verdict, and summary.json.
# ---------------------------------------------------------------------------------------------------------------
# News 0: with no news there is nothing to react to, so the trained model should play like the baseline.
# Clearly below 0 = it learned to add days everywhere regardless of the news.
baseline_news0 = [E.run_episode("baseline", t, s, news=0, scenario="random") for t in TASKS for s in check_seeds]
news0 = E.paired_comparison(trained_news0, baseline_news0)["ALL"]
# Fixed scenario: the original tasks. Training used the random scenario; it should not have made these worse.
fixed = E.paired_comparison(trained_fixed, base_fixed)["ALL"]
print(line("news 0: trained - baseline", {"ALL": news0}), " (should be close to 0: no news, nothing to add)")
print(line("fixed scenario: trained - base", {"ALL": fixed}), " (should not be clearly below 0)")
print(f"parse failures per episode, trained: {st.mean(r['parse_failures'] for r in trained_eps):.1f}, "
      f"base: {st.mean(r['parse_failures'] for r in base_eps):.1f}")

# The verdict. Success needs both:
#   1. trained beats base, with the 95% interval above 0 (not luck)
#   2. trained beats "+3 everywhere" (it reads the news, not just stocks more)
a, b = vs_base["ALL"], vs_plus3["ALL"]
signal = a["mean"] - a["ci95"] > 0
beats_plus3 = b["mean"] > 0
print("\n==== VERDICT ====")
print(f"trained beats base (interval above 0): {'YES' if signal else 'no'}  ({a['mean']:+.4f} ± {a['ci95']:.4f})")
print(f"trained beats +3 everywhere:           {'YES' if beats_plus3 else 'no'}  ({b['mean']:+.4f} ± {b['ci95']:.4f})")
print(f"test probe, trained - base:            {probe_diff['mean']:+.3f} ± {probe_diff['ci95']:.3f} points")
if signal and beats_plus3:
    print("-> first positive signal: the model learned to use the news.")
elif signal:
    print("-> better than the base model, but not yet better than simply adding 3 days everywhere.")
else:
    print("-> no clear signal yet. Check the probe curve and the health metrics (part 6) to see why.")

# Everything important in one file, to share or to commit to the repo's results/ folder.
summary = {
    "config": {k: v for k, v in settings.items() if isinstance(v, (int, float, str, bool))},
    "mean_scores": {n: E.summarize(r)["ALL"]["mean"] for n, r in scores.items()},
    "trained_vs_base": vs_base, "trained_vs_plus3": vs_plus3, "news0_vs_baseline": news0,
    "fixed_vs_base": fixed, "test_probe_base": base_test_probe_summary,
    "test_probe_trained": trained_test_probe_summary, "test_probe_diff": probe_diff,
    "probe_history": ph,
}
json.dump(summary, open(OUT / "summary.json", "w"), indent=1, default=float)

## 9. A look at the answers, GGUF export, results zip

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Part 9a: a look at the answers. The 6 test-probe days where training changed the reward the most, with the
# news, the base answer and the trained answer. Useful to see *what* the model learned (or broke).
# ---------------------------------------------------------------------------------------------------------------
changes = sorted(zip(test_probe_items, base_test_probe, trained_test_probe),
                 key=lambda x: abs(x[2]["reward"] - x[1]["reward"]), reverse=True)
for item, b, t in changes[:6]:
    news = " / ".join(build_state(item.prompt)[1].news)       # the news visible on that day
    print(f"--- {item.prompt.task}, seed {item.prompt.seed}, day {item.prompt.day} | news SKUs {item.skus} | "
          f"best reference {item.best:+.2f}")
    print("  news:", news[:300])
    print(f"  base    {b['reward']:+.2f}  {b['text'][:150]!r}")
    print(f"  trained {t['reward']:+.2f}  {t['text'][:150]!r}")

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Part 9b: GGUF export for Ollama (about 10-20 min: it builds llama.cpp and quantises).
# It merges the LoRA into the weights (W + B·A) and saves one 4-bit file that Ollama can run on a laptop.
#
# Then, on your computer, in the folder with the downloaded files:
#   ollama create pharma-phi4-mini-grpo -f Modelfile
#   uv run python local_llm/run_local_llm.py --server-model pharma-phi4-mini-grpo --news 2 --random-scenarios
# ---------------------------------------------------------------------------------------------------------------
if EXPORT_GGUF:
    try:
        model.save_pretrained_gguf(str(OUT / "phi4mini_pharma_gguf"), tokenizer, quantization_method="q4_k_m")
        gguf = sorted((OUT / "phi4mini_pharma_gguf").glob("*.gguf"))
        print("GGUF:", gguf)
        if gguf:
            # Ollama's recipe: this GGUF file, with a context window large enough for our ~2,600-token prompts.
            (OUT / "phi4mini_pharma_gguf" / "Modelfile").write_text(f"FROM ./{gguf[-1].name}\nPARAMETER num_ctx 8192\n")
    except Exception as exc:
        # Not fatal: the LoRA adapter is saved, and the GGUF can be made from it later.
        print("GGUF export failed (the LoRA adapter is saved; it can be exported later):", exc)

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Part 9c: one zip to download (Kaggle: Output -> grpo_results.zip): logs, results, plots, worker logs and the LoRA
# adapter. Checkpoints and the GGUF are left out (large); they stay in /kaggle/working/grpo_run.
# ---------------------------------------------------------------------------------------------------------------
import shutil

bundle = Path(WORK) / "grpo_results"
bundle.mkdir(exist_ok=True)
for f in OUT.iterdir():
    if f.is_file():                     # files only: skips the checkpoints/, LoRA and GGUF folders
        shutil.copy(f, bundle / f.name)
shutil.copytree(ADAPTER_DIR, bundle / ADAPTER_DIR.name, dirs_exist_ok=True)
print(shutil.make_archive(str(bundle), "zip", root_dir=bundle))